# Load Packages

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import torch

# Import Functions
sys.path.append("../../")

from matplotlib import pyplot as plt
from tqdm.auto import tqdm

from os.path import join
import sys
sys.path.append("../") # Add directory containing src to path

from src.configs.octmnist_config import data_name, data_name_oods, batch_size, eval_batch_size
from src.file_manager.filepath import FilePath
from src.models.postnet.result_processing import process_pn_results
from src.file_manager.load_save_df import load_pred_df, save_pred_perf_df
from src.evaluation.evaluate import get_model_performance
from src.file_manager.load_save_model import load_model

from src.models.dec.model import DEC
from src.models.dec.train import train_dec
from src.models.dec.predict import get_dec_model_prediction
from src.evaluation.inference import get_all_predictions
from src.file_manager.load_save_df import save_pred_df

from src.data_generator.oct_mnist import load_octmnist_data_dict
from src.data_generator.chest_mnist import load_chestmnist_data_dict
from src.data_generator.octdl import load_octdl_data_dict
from src.data_processing.ood_dataset_preprocessing import process_dataset_for_ood, left_join_datasets

from cur_seed import seed
# seed = 2024

fp = FilePath(data_name=data_name, seed=seed)
fp_ood = FilePath(data_name=data_name_oods[0], seed=seed)
fp_ood2 = FilePath(data_name=data_name_oods[1], seed=seed)

# Load Data

In [ ]:
data_dict = load_octmnist_data_dict(fp_preprocessed=fp.get_preprocessed_folder())
num_ori_test = len(data_dict["test_df"])
data_dict_ood = load_chestmnist_data_dict(fp_preprocessed=fp_ood.get_preprocessed_folder(), only_test=True)
data_dict_ood = process_dataset_for_ood(data_dict, data_dict_ood, seed)
octdl_in_data_dict, octdl_out_data_dict = load_octdl_data_dict(
    fp_preprocessed=fp_ood2.get_preprocessed_folder())
data_dict = left_join_datasets(data_dict, octdl_in_data_dict)
octdl_in_data_dict = process_dataset_for_ood(data_dict, octdl_in_data_dict, seed)
octdl_out_data_dict = process_dataset_for_ood(data_dict, octdl_out_data_dict, seed)

# Training

In [ ]:
best_param = {"feat_extractor":"resnet"}
dec_model, _ = train_dec(
    best_param, data_dict, seed, batch_size,
    max_epochs=500, patience=5, freeze_layers=False, reg_weight=0.01
)
fp_model = fp.get_fp_model(DEC, cur_model_name="tuned")
torch.save(dec_model, fp_model)

# Prediction

In [ ]:
dec_model = load_model(fp=fp, ModelClass=DEC, cur_model_name="tuned")
pred_df = get_all_predictions(
    model=dec_model, 
    data_dict=data_dict, 
    batch_size=batch_size, 
    eval_batch_size=eval_batch_size,
    pred_func=get_dec_model_prediction,
    seed=seed,
)
pred_df["split_perf"] = pred_df["split"]
pred_df["split_perf"][pred_df["split"]=="Test"] = ["Test-OCTMNIST" for i in range(num_ori_test)] + \
    ["Test-OCTDL" for i in range((pred_df["split"]=="Test").sum()-num_ori_test)]
save_pred_df(pred_df=pred_df, fp=fp, ModelClass=DEC)


# Performance Evaluation

In [ ]:
pred_df = load_pred_df(fp=fp, ModelClass=DEC)
perf_df = get_model_performance(
    all_pred_df=pred_df, data_dict=data_dict, label="dec", perf_split_col="split_perf")
save_pred_perf_df(pred_perf_df=perf_df, fp=fp, ModelClass=DEC)
perf_df

# OOD Prediction

In [ ]:
bnn_model = load_model(fp=fp, ModelClass=DEC, cur_model_name="tuned")
ood_dicts = {
    "ood_in_octdl": octdl_in_data_dict, 
    "ood_out_octdl": octdl_out_data_dict,
    "ood_chestmnist": data_dict_ood}
for label, cur_ood_data_dict in tqdm(ood_dicts.items(), total=len(ood_dicts)):
    pred_df_ood = get_all_predictions(
        model=bnn_model, 
        data_dict=cur_ood_data_dict, 
        batch_size=batch_size, 
        eval_batch_size=eval_batch_size,
        pred_func=get_dec_model_prediction,
        seed=seed,
        additional_pred_args={"ood": True},
    )
    save_pred_df(pred_df=pred_df_ood, fp=fp, ModelClass=DEC, optional_label=label)